# 4.2 Adaptivity Fields

Global mesh adaptivity is set on `MeshManager`. Local material properties such as `vadapt`, `epw_mult`, `hmin`, and `hmax` can guide refinement in specific layers.

By the end, you should understand how global adaptivity settings and layer-level sizing fields work together without changing physical material properties.


## How To Read This Tutorial

Adaptivity fields are local hints that shape how the solver refines a mesh. They are not material physics. Use them to tell the mesher where small elements are required, where large elements are acceptable, and how wave-resolution targets should vary across the model.

This notebook keeps the physics small and focuses on the contract: author `hmin`, `hmax`, and `epw_mult` style controls, submit a frequency-domain QC job, and inspect the resulting mesh output.

## Design Notes

Adaptivity fields are material/model hints that influence solver-side refinement. They are different from physical material properties: they do not change wave speed or density, but they do tell the mesher how aggressive it may be in different regions.

| Field/control | Meaning | Typical use |
| --- | --- | --- |
| `elems_per_wave` | Global elements-per-wavelength target. | Coarse first control for wavefield resolution. |
| `order` | Polynomial order used by the adapted solve/output. | Higher order can reduce elements for smooth fields. |
| `f_low`, `f_high` | Frequency range used for sizing. | Match the job band so mesh sizing is not guessed. |
| `hmin` | Smallest allowed element size. | Prevents runaway refinement near small features. |
| `hmax` | Largest allowed element size. | Prevents under-resolution in slow or important zones. |
| `epw_mult` | Local multiplier on wavefield sizing. | Tightens or relaxes resolution per layer/property region. |

This notebook writes ParaView output so the resulting mesh controls can be checked visually instead of treated as hidden solver settings.


## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display
import frequensolve as fs

u = fs.ureg


## Layer-Level Adaptivity Hints

The upper layer below carries local mesh hints in the same property dictionary as the physical model. That keeps the intent spatially attached to the material region: the slow, shallow layer can request different sizing than the faster basement without changing the global mesh policy.

`epw_mult` locally scales the elements-per-wavelength target, `vadapt` supplies the speed used for adaptivity sizing, and `hmin`/`hmax` bound element sizes. These are mesh controls rather than material properties. They should be documented near the model because they affect cost and resolution, but they should not be confused with `Vp`, `Rho`, or other physics-defining properties.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="adaptivity_fields",
    path="./scratch/adaptivity_fields",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="adaptivity_fields",
    physics="acoustic",
    dimension=2,
    units={
        "length": "km",
        "velocity": "km/s",
        "density": "g/cm^3",
    },
)

model = fs.LayeredModel(
    name="model",
    dimension=2,
    x_limits=[0.0, 1.0],
)
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(
    name="cap",
    properties={
        "Vp": 1.5 * u.km / u.s,
        "Rho": 1.0 * u.g / u.cm**3,
        "epw_mult": 1.5,
        "hmax": 0.04 * u.km,
    },
)
model.add_surface(name="interface", depth=0.2 * u.km)
model.add_layer(
    name="basement",
    properties={
        "Vp": 2.6 * u.km / u.s,
        "Rho": 2.2 * u.g / u.cm**3,
        "vadapt": 1.4 * u.km / u.s,
        "hmin": 0.01 * u.km,
    },
)
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model
model.plot("vp", figsize=(7, 3), aspect="equal")


## Submit A ParaView Mesh Output Job

The job asks for both wavefield output and mesh-control properties. That combination is deliberate: the pressure field shows the solve result at one frequency, while `epw_mult`, `hmin`, `hmax`, and `Subdomain` make the sizing intent visible in the same VTK file.

The run cell is strict. If the solver-side adaptivity machinery rejects a field or fails during refinement, the notebook should stop with the job logs available rather than silently substituting a simpler mesh.


In [ ]:
sim += model.hex_mesh_generator([4, 4])
sim.mesh.set_adapt(
    elems_per_wave=2.0,
    order=4,
    f_low=5.0,
    f_high=35.0,
    hmin=0.005,
    hmax=0.08,
)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(
    conditions=["pml"],
    boundaries=["x_min", "x_max", "z_max"],
    pml_wavelengths=0.5,
)

acq = fs.Acquisition()
acq.add_source_group(kind="scalar", coords=[[0.5, 0.05]])
node = fs.ReceiverNode(name="hydrophone")
node.add_component(name="p", field="pressure")
acq.add_receiver_group(
    name="surface",
    device=node,
    coords=[[x, 0.025] for x in np.linspace(0.1, 0.9, 21)],
)
sim += acq
sim += fs.Discretization()

site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
job = fs.FrequencyDomainJob(
    name="freq_adaptivity",
    simulation=sim,
    f_list=[25.0],
    outputs=[
        fs.ParaviewOutput(
            name="adaptivity",
            fields=["pressure"],
            properties=["vp", "epw_mult", "hmin", "hmax", "Subdomain"],
            show_pml=True,
            upscale=1,
            order=2,
        )
    ],
)
result = site.submit(job).wait()


## Discover ParaView Files

The result object records files produced by the job. Filtering by base name and suffix keeps the notebook independent of the exact project directory layout.


In [ ]:
vtu_files = result.output_files(base="adaptivity", suffix=".vtu", existing=True)
[str(path) for path in vtu_files]


## Render Persistent Mesh Screenshots

These cells use the generated VTK output as evidence for the meshing controls. The screenshots are written to `assets/` so the notebook remains useful after the PyVista process is gone.

The two rendered views are intentionally different. A property view confirms that the exported material/control field has the intended spatial support. An edge view confirms how that intent affected the generated mesh. For release examples, both are worth keeping: one explains what was requested, the other shows what was built.


In [ ]:
image_dir = Path("./assets")
image_dir.mkdir(exist_ok=True)
rendered = []
for field, filename in [('vp', 'adaptivity_vp.png'), ('hmin', 'adaptivity_hmin.png')]:
    screenshot = image_dir / filename
    fs.plot_vtu(
        vtu_files[0],
        field=field,
        show_edges=True,
        scalar_bar=True,
        show=False,
        screenshot=screenshot,
        window_size=(1100, 500),
    )
    rendered.append(screenshot)

for screenshot in rendered:
    display(Image(filename=str(screenshot)))


## Before Moving On

The goal is controlled refinement, not a uniformly tiny mesh. Review where the mesh is fine, where it is coarse, and whether those choices match the physics and geometry. If the mesh surprises you, inspect the exported adaptivity fields before changing solver settings.

A good production habit is to begin with interpretable adaptivity hints, inspect a frequency-domain mesh/output job, and only then scale to expensive sweeps.

## Result Review Checklist

Use the screenshots as evidence for two separate questions: did the requested fields export, and did the adapted mesh respond in the intended region? A pretty pressure image alone is not enough for mesh QA.

| View | Review question |
| --- | --- |
| `vp` with edges | Does the mesh resolve the wavefield and layer geometry without globally over-refining? |
| `hmin` or `hmax` | Are local sizing bounds present only on the layers where they were authored? |
| `Subdomain` in ParaView | Do material labels match the layer names/properties you expected? |
| Job logs | Did adaptivity accept the field names, units, and global `set_adapt(...)` controls? |

If the mesh is unexpectedly dense everywhere, lower-level controls such as `hmin`, `hmax`, or `epw_mult` are often too aggressive or attached to too large a region.
